# SO-101 Cartesian7 / MuJoCo 3D (Google Drive)

`MyDrive/SO101_Cartesian7/so101_colab_assets.zip` からURDF/mesh/IK候補を取得します。GitHub連携・実機接続なし。3D MP4動画と回転可能なPlotly 3D表示を作ります。**100個のIK候補は独立したオフライン推定であり、閉ループのロボット動作や安全性を実証するものではありません。**

In [ ]:
import os
os.environ['MUJOCO_GL']='egl'
!pip -q install 'mujoco>=3.3,<4' 'imageio[ffmpeg]' plotly matplotlib
import mujoco, numpy as np, json, zipfile, imageio
from pathlib import Path
from IPython.display import Video, display
from google.colab import drive
drive.mount('/content/drive')
print('MuJoCo',mujoco.__version__)

## 入力設定

Google Driveのフォルダ名を変更した場合のみ `DRIVE_FOLDER` を編集。GPUランタイム（T4など）を推奨。

In [ ]:
DRIVE_FOLDER=Path('/content/drive/MyDrive/SO101_Cartesian7')
ZIP_FILENAME='so101_colab_assets.zip'
URDF_RELATIVE_PATH='data/urdf/so101/so101_new_calib.urdf'
IK_RELATIVE_PATH='ik_candidates.json'
assert (DRIVE_FOLDER/ZIP_FILENAME).is_file(), 'ZIPが見つかりません。Driveアカウントとフォルダ名を確認。'
ROOT=Path('/content/so101_mujoco_workspace');ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(DRIVE_FOLDER/ZIP_FILENAME) as z:
 for item in z.infolist():
  target=(ROOT/item.filename).resolve()
  if not target.is_relative_to(ROOT.resolve()):raise ValueError('Unsafe zip path')
 z.extractall(ROOT)
URDF=ROOT/URDF_RELATIVE_PATH
IK_PATH=ROOT/IK_RELATIVE_PATH
assert URDF.exists(),f'URDF not found: {URDF}'
assert IK_PATH.exists(),f'IK not found: {IK_PATH}'
OUTPUT_DIR=DRIVE_FOLDER/'mujoco_results';OUTPUT_DIR.mkdir(exist_ok=True)
print(URDF,IK_PATH,OUTPUT_DIR,sep='\n')

## URDFロード・IK候補・モデル内接触

In [ ]:
model=mujoco.MjModel.from_xml_path(str(URDF.resolve()))
data=mujoco.MjData(model)
NAMES=['shoulder_pan','shoulder_lift','elbow_flex','wrist_flex','wrist_roll']
jids=[mujoco.mj_name2id(model,mujoco.mjtObj.mjOBJ_JOINT,n) for n in NAMES]
assert all(j>=0 for j in jids)
addrs=[int(model.jnt_qposadr[j]) for j in jids]
items=json.loads(IK_PATH.read_text())
angles=np.asarray([item['candidate_motor_deg'] for item in items],float)
assert angles.ndim==2 and angles.shape[1]==5 and np.isfinite(angles).all()
def apply_pose(p):
 data.qpos[:]=model.qpos0
 for adr,v in zip(addrs,p):data.qpos[adr]=np.deg2rad(v)
 mujoco.mj_forward(model,data)
def contacts():
 pairs={}
 for i in range(data.ncon):
  cc=data.contact[i]
  names=[]
  for geom in (int(cc.geom1),int(cc.geom2)):
   b=int(model.geom_bodyid[geom]);names.append(mujoco.mj_id2name(model,mujoco.mjtObj.mjOBJ_BODY,b) or f'body#{b}')
  k=' ↔ '.join(sorted(names)); pairs.setdefault(k,[]).append(float(cc.dist))
 return {k:{'count':len(ds),'min_distance_mm':round(min(ds)*1000,3)} for k,ds in pairs.items()}
apply_pose(np.zeros(5));neutral_pairs=contacts()
frames=[];bodyposes=[]
for pose in angles:
 apply_pose(pose);frames.append(contacts());bodyposes.append(np.asarray(data.xpos).copy())
bodyposes=np.asarray(bodyposes)
print('bodies',model.nbody,'geoms',model.ngeom,'IK candidates',len(angles),'neutral contacts',neutral_pairs)
print('first candidate contacts',frames[0])

## MuJoCo 3D動画（MP4）

URDF由来のMuJoCoジオメトリをオフスクリーンでレンダリングします。macOSのビューアーは使用しません。EGLが利用できない場合はGPUランタイムを選択して再実行。

In [ ]:
video_path=OUTPUT_DIR/'cartesian7_mujoco_3d.mp4'
cam=mujoco.MjvCamera();cam.type=mujoco.mjtCamera.mjCAMERA_FREE
cam.lookat=np.array([.08,0,.13]);cam.distance=.68;cam.azimuth=140;cam.elevation=-25
try:
 with mujoco.Renderer(model,height=480,width=640) as renderer:
  with imageio.get_writer(str(video_path),fps=10,codec='libx264',quality=7) as writer:
   for i,pose in enumerate(angles):
    apply_pose(pose);renderer.update_scene(data,camera=cam);writer.append_data(renderer.render())
 display(Video(str(video_path),embed=True,html_attributes='controls loop'))
 print('Saved MP4:',video_path)
except Exception as exc:
 print('Renderer error:',repr(exc),'— GPUランタイムを確認。下のPlotly 3DはOpenGL不要。')

## 操作できる3D表示（回転・ズーム・フレーム切替）

これはリンク中心の**骨格**であり、実際の衝突メッシュではありません。

In [ ]:
import plotly.graph_objects as go
parents=np.asarray(model.body_parentid)
edges=[(int(parents[j]),j) for j in range(1,model.nbody) if parents[j]>0]
def plot_trace(i):
 p=bodyposes[i];xx=[];yy=[];zz=[]
 for u,v in edges:
  xx.extend([p[u,0],p[v,0],None]);yy.extend([p[u,1],p[v,1],None]);zz.extend([p[u,2],p[v,2],None])
 return go.Scatter3d(x=xx,y=yy,z=zz,mode='lines',line=dict(width=8,color='#2d7db8'))
fs=[go.Frame(data=[plot_trace(i)],name=str(i)) for i in range(len(angles))]
fig=go.Figure(data=[plot_trace(0)],frames=fs)
fig.update_layout(height=650,title='SO-101 Cartesian7 / 3D IK candidates',scene=dict(aspectmode='data',xaxis_title='X(m)',yaxis_title='Y(m)',zaxis_title='Z(m)'),updatemenus=[dict(type='buttons',buttons=[dict(label='Play',method='animate',args=[None,dict(frame=dict(duration=100,redraw=True),transition=dict(duration=0))]),dict(label='Pause',method='animate',args=[[None],dict(mode='immediate',frame=dict(duration=0))])])],sliders=[dict(currentvalue=dict(prefix='Frame: '),steps=[dict(label=str(i),method='animate',args=[[str(i)],dict(mode='immediate',frame=dict(duration=0,redraw=True))]) for i in range(len(angles))])])
html=OUTPUT_DIR/'cartesian7_interactive_3d.html';fig.write_html(str(html),include_plotlyjs=True);fig.show();print('Saved:',html)

## 解析結果（同じGoogle Driveフォルダへ保存）

**注意:** 机・ムヒ・容器の座標は未校正。リンク間接触の検出はモデルジオメトリに依存し、実機での衝突を証明しません。100件は独立推定点で補間軌道検査は未実施。

In [ ]:
from collections import Counter
counts=Counter(k for f in frames for k in f)
report={'status':'SIMULATION_ONLY_NOT_HARDWARE_APPROVED','mujoco_version':mujoco.__version__,'frames':len(angles),'neutral_contact_pairs':neutral_pairs,'contact_pair_frame_counts':dict(counts),'first_candidate_contact_pairs':frames[0],'warnings':['Uncalibrated LeRobot to URDF joint mapping','No real desk / Muhi / cup geometry','No swept path analysis','No physical collision approval']}
out=OUTPUT_DIR/'summary_3d.json';out.write_text(json.dumps(report,ensure_ascii=False,indent=2));print(out.read_text());print('Saved:',out)